## Value-requires-Statement Repair Analyzer

In [ ]:

import pandas as pd

# Read the first CSV file into a DataFrame
df_vrs_repairs = pd.read_csv("vrs_correcoes.csv")

df_vrs_repairs

- The cell below counts different types of basic T-box repairs generated with the relational database:

In [ ]:
print(len(df_vrs_repairs[(df_vrs_repairs['C_deleted'] == True)] ))
print(len(df_vrs_repairs[(df_vrs_repairs['C_deprecated'] == True)] ))
print(len(df_vrs_repairs[(df_vrs_repairs['CQ_added_exception'] == True)] ))

- checking for base statement deletion:

In [ ]:
df_vrs_repairs['S_deleted'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def statementDeleted(row):
    
    # blank node, test later
    if row['object'].startswith("_:"):
        return None
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{wdt_pid}> <{row['object']}>  }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
print(statementDeleted(df_vrs_repairs.iloc[0]))

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs.iterrows(), total=len(df_vrs_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['S_deleted']):
        result = statementDeleted(row)
        df_vrs_repairs.at[index, 'S_deleted'] = result

    # Save a checkpoint every 10,000 rows
    if index % 10000 == 0:
        df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_vrs_repairs

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['S_deleted'] == True)] )

- testing for context statement addition

In [ ]:
df_vrs_repairs['Sc_added'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def onlyReqPropStmtAdded(row):
    
    if row['object'].startswith("_:"):
        return None
    
    if row['2019_no_req_val'] is False:
        return None
    
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['object']}> <{row['wdt_required_property']}> []}}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
print(onlyReqPropStmtAdded(df_vrs_repairs.iloc[0]))

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getRequiredValues(row, endpoint = "ENTER_qEndpoint_WD_2019"):
    
    if row['object'].startswith("_:"):
        return None
    
    if row['2019_no_req_val'] is True:
        return None
    
    wd_required_pid = row['wdt_required_property'].replace("http://www.wikidata.org/prop/direct/", "http://www.wikidata.org/entity/")
    
    # SPARQL query
    query = f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX wikibase: <http://wikiba.se/ontology#>
        PREFIX p: <http://www.wikidata.org/prop/>
        PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
        PREFIX ps: <http://www.wikidata.org/prop/statement/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>

        SELECT 
          ?value
        WHERE
        {{
          <{row['property']}> p:P2302 ?statement.
          ?statement ps:P2302 wd:Q21510864.
          ?statement pq:P2306 <{wd_required_pid}>.
          ?statement pq:P2305 ?value. 
        }}
    """
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find all 'uri' elements and extract their text
        uris = [uri_element.text for uri_element in root.findall('.//ns:uri', namespace)]

        if len(uris) == 0:
            return []
        
        return uris
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
    
def reqPropValStmtAdded(row, required_value):
    
    if row['object'].startswith("_:"):
        return None
    
    if row['2019_no_req_val'] is True:
        return None
    
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['object']}> <{row['wdt_required_property']}> <{required_value}>}}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
def testReqStmtWithReqValueAdded(row):
    reqValues = getRequiredValues(row)
    if reqValues is None:
        return None
    if len(reqValues) == 0:
        return None
    for reqVal in reqValues:
        result = reqPropValStmtAdded(row, reqVal)
        if result is True:
            return True
    return False

testReqStmtWithReqValueAdded(df_vrs_repairs.iloc[482273])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs.iterrows(), total=len(df_vrs_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['Sc_added']):
        if row['2019_no_req_val'] is True:
            result = onlyReqPropStmtAdded(row)
            df_vrs_repairs.at[index, 'Sc_added'] = result
        else:
            result = testReqStmtWithReqValueAdded(row)
            df_vrs_repairs.at[index, 'Sc_added'] = result

    # Save a checkpoint every 10,000 rows
    if index % 300000 == 0:
        df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_vrs_repairs.to_csv("checkpoint_vrs_ok.csv", index=False)

- checking rows still to be classified:

In [ ]:
df_vrs_repairs[
    (df_vrs_repairs['C_deleted'] == False)
    & (df_vrs_repairs['C_deprecated'] == False)
    & (df_vrs_repairs['CQ_added_exception'] == False)
    & (df_vrs_repairs['S_deleted'] == False)
    & ((df_vrs_repairs['Sc_added'] == False) 
       | (pd.isna(df_vrs_repairs['Sc_added'])))
]

- revalidating constraint deprecation:

In [ ]:
import requests
import xml.etree.ElementTree as ET

def hasDeprecatedReason(row):

    if bool(row['2019_no_req_val']):
        string_req_val = "FILTER NOT EXISTS {{?statement pq:P2305 []}}"
    else:
        string_req_val = "?statement pq:P2305 []."
        

    # SPARQL query
    query = f"""PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX wikibase: <http://wikiba.se/ontology#>
            PREFIX p: <http://www.wikidata.org/prop/>
            PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
            PREFIX ps: <http://www.wikidata.org/prop/statement/>
            PREFIX wd: <http://www.wikidata.org/entity/>
            PREFIX wdt: <http://www.wikidata.org/prop/direct/>

            ASK
            {{
              ?statement ps:P2302 wd:Q21510864. ## value-requires-statement constraint
              <{row['property']}> p:P2302 ?statement.
              ?statement pq:P2306/wikibase:directClaim <{row['wdt_required_property']}>.

              {string_req_val}
              # exception and deprecation
              ?statement pq:P2241 ?rank
            }}
            """
    #print(query)
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
hasDeprecatedReason(df_vrs_repairs.iloc[8096])

In [ ]:
type(df_vrs_repairs.iloc[8096]['C_deprecated'])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs.iterrows(), total=len(df_vrs_repairs)):
    
    # Check for unprocessed rows
    if bool(row['C_deprecated']) is False:
        result = hasDeprecatedReason(row)
        df_vrs_repairs.at[index, 'C_deprecated'] = result
       

    # Save a checkpoint every 10,000 rows
    if index % 100000 == 0:
        df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['C_deprecated'] == True)] )

- checking rows still to be validated:

In [ ]:
df_vrs_repairs[
    (df_vrs_repairs['C_deleted'] == False)
    & (df_vrs_repairs['C_deprecated'] == False)
    & (df_vrs_repairs['CQ_added_exception'] == False)
    & (df_vrs_repairs['S_deleted'] == False)
    & ((df_vrs_repairs['Sc_added'] == False) 
       | (pd.isna(df_vrs_repairs['Sc_added'])))
]

- test deletion of base statement with blank node as object:

In [ ]:
import requests
import xml.etree.ElementTree as ET

def statementWithBlankObjDeleted(row):
    
    if not row['object'].startswith("_:"):
        return None
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{wdt_pid}> []}}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
print(statementWithBlankObjDeleted(df_vrs_repairs.iloc[1392179]))

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs.iterrows(), total=len(df_vrs_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['S_deleted']) or row['object'].startswith("_:"):
        result = statementWithBlankObjDeleted(row)
        df_vrs_repairs.at[index, 'S_deleted'] = result

    # Save a checkpoint every 10,000 rows
    if index % 300000 == 0:
        df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['S_deleted'] == True)] )

In [ ]:
df_vrs_repairs[
    (df_vrs_repairs['C_deleted'] == False)
    & (df_vrs_repairs['C_deprecated'] == False)
    & (df_vrs_repairs['CQ_added_exception'] == False)
    & (df_vrs_repairs['S_deleted'] == False)
    & ((df_vrs_repairs['Sc_added'] == False) 
       | (pd.isna(df_vrs_repairs['Sc_added'])))
]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def hasStmtWithBlankNode(row, endpoint):
    
    if not row['object'].startswith("_:"):
        return None
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    #endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{wdt_pid}> ?o.
            FILTER(!isIRI(?o) || !STRSTARTS(STR(?o), "http://"))
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
def isStmtWithBlankNodeRemoved(row):
    if hasStmtWithBlankNode(row, "ENTER_qEndpoint_WD_2019") and not hasStmtWithBlankNode(row, "ENTER_qEndpoint_WD_2023"):
        return True
    return False

# Example usage
#print(hasStmtWithBlankNode(df_vrs_repairs.iloc[179], "ENTER_qEndpoint_WD_2019"))
#print(hasStmtWithBlankNode(df_vrs_repairs.iloc[179], "ENTER_qEndpoint_WD_2023"))
isStmtWithBlankNodeRemoved(df_vrs_repairs.iloc[179])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs.iterrows(), total=len(df_vrs_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['S_deleted']) or row['object'].startswith("_:"):
        result = isStmtWithBlankNodeRemoved(row)
        df_vrs_repairs.at[index, 'S_deleted'] = result

    # Save a checkpoint every 10,000 rows
    if index % 300000 == 0:
        df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['S_deleted'] == True)] )

In [ ]:
df_vrs_repairs[
    (df_vrs_repairs['C_deleted'] == False)
    & (df_vrs_repairs['C_deprecated'] == False)
    & (df_vrs_repairs['CQ_added_exception'] == False)
    & (df_vrs_repairs['S_deleted'] == False)
    & ((df_vrs_repairs['Sc_added'] == False) 
       | (pd.isna(df_vrs_repairs['Sc_added'])))
]

- Now, revalidate blank nodes because they might still be violations (as the HDT server assigned an id to the blank nodes):

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isBlankStillViolation(row):
    
    if not row['object'].startswith("_:"):
        return None
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{wdt_pid}> ?o.
            ?o <{row['wdt_required_property']}> []
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
# Example usage
isBlankStillViolation(df_vrs_repairs.iloc[206])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

blank_violations = []
# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs.iterrows(), total=len(df_vrs_repairs)):
    
    # Check for unprocessed rows
    if row['object'].startswith("_:"):
        if (row['C_deleted'] == False) & (row['C_deprecated'] == False) & (row['CQ_added_exception'] == False) & (row['S_deleted'] == False)    & ((row['Sc_added'] == False)        | (pd.isna(row['Sc_added']))):
            if isBlankStillViolation(row):
                blank_violations.append(index)

In [ ]:
blank_violations[:20]

In [ ]:
len(blank_violations)

In [ ]:
len(df_vrs_repairs)

- cleaning repairs set by removing blank nodes violations:

In [ ]:
# Assuming df_vrs_repairs is your DataFrame and blank_violations is your list of indexes
df_vrs_repairs_cleaned = df_vrs_repairs.drop(index=blank_violations)

# Optionally, you can reset the index if needed
df_vrs_repairs_cleaned.reset_index(drop=True, inplace=True)

In [ ]:
len(df_vrs_repairs_cleaned)

In [ ]:
df_vrs_repairs_cleaned[
    (df_vrs_repairs_cleaned['C_deleted'] == False)
    & (df_vrs_repairs_cleaned['C_deprecated'] == False)
    & (df_vrs_repairs_cleaned['CQ_added_exception'] == False)
    & (df_vrs_repairs_cleaned['S_deleted'] == False)
    & ((df_vrs_repairs_cleaned['Sc_added'] == False) 
       | (pd.isna(df_vrs_repairs_cleaned['Sc_added'])))
]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isBlankStillViolation2(row):
    
    if not row['object'].startswith("_:"):
        return None
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{wdt_pid}> ?o.
            FILTER NOT EXISTS {{?o <{row['wdt_required_property']}> []}}
            FILTER(!isIRI(?o))
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
# Example usage
isBlankStillViolation2(df_vrs_repairs_cleaned.iloc[286])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

blank_violations_2 = []
# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs_cleaned.iterrows(), total=len(df_vrs_repairs_cleaned)):
    
    # Check for unprocessed rows
    if row['object'].startswith("_:"):
        if (row['C_deleted'] == False) & (row['C_deprecated'] == False) & (row['CQ_added_exception'] == False) & (row['S_deleted'] == False)    & ((row['Sc_added'] == False)        | (pd.isna(row['Sc_added']))):
            if isBlankStillViolation2(row):
                blank_violations_2.append(index)

In [ ]:
blank_violations_2[:10]

In [ ]:
df_vrs_repairs_cleaned.iloc[8719]

In [ ]:
# Assuming df_vrs_repairs is your DataFrame and blank_violations is your list of indexes
df_vrs_repairs_cleaned = df_vrs_repairs_cleaned.drop(index=blank_violations_2)

# Optionally, you can reset the index if needed
df_vrs_repairs_cleaned.reset_index(drop=True, inplace=True)

In [ ]:
df_vrs_repairs_cleaned

In [ ]:
df_vrs_repairs_cleaned[
    (df_vrs_repairs_cleaned['C_deleted'] == False)
    & (df_vrs_repairs_cleaned['C_deprecated'] == False)
    & (df_vrs_repairs_cleaned['CQ_added_exception'] == False)
    & (df_vrs_repairs_cleaned['S_deleted'] == False)
    & ((df_vrs_repairs_cleaned['Sc_added'] == False) 
       | (pd.isna(df_vrs_repairs_cleaned['Sc_added'])))
]

- Test for t-box repair required property replacement (CQr):

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getConstraintID(row, endpoint = "ENTER_qEndpoint_WD_2019"):
        
    #wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    #endpoint = "ENTER_qEndpoint_WD_2019"

    if bool(row['2019_no_req_val']):
        req_val = "FILTER NOT EXISTS {?statement pq:P2305 []}"
    else:
        req_val = "?statement pq:P2305 []."
    
    # SPARQL query
    query = f"""
            PREFIX psv: <http://www.wikidata.org/prop/statement/value/>
            PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX wikibase: <http://wikiba.se/ontology#>
            PREFIX p: <http://www.wikidata.org/prop/>
            PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
            PREFIX ps: <http://www.wikidata.org/prop/statement/>
            PREFIX wd: <http://www.wikidata.org/entity/>
            PREFIX wdt: <http://www.wikidata.org/prop/direct/>

            SELECT ?statement {{
              ?statement ps:P2302 wd:Q21510864. ## value-requires-statement constraint
              <{row['property']}> p:P2302 ?statement.
              ?statement pq:P2306/wikibase:directClaim <{row['wdt_required_property']}>.
              # exception and deprecation
              FILTER NOT EXISTS {{?statement pq:P2241 []}}
              FILTER NOT EXISTS {{?statement wikibase:rank wikibase:DeprecatedRank}}
              {req_val}
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        #print(response.text)

        # Parse the XML data
        namespace = {'sparql': 'http://www.w3.org/2005/sparql-results#'}
        root = ET.fromstring(response.text)

        # Find the uri inside the binding
        if root.find('.//sparql:binding[@name="statement"]/sparql:uri', namespace) is not None:
            return root.find('.//sparql:binding[@name="statement"]/sparql:uri', namespace).text
        return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
    
def getRequiredProperty(constraint_statement):
        
    #wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"
    
    # SPARQL query
    query = f"""
            PREFIX psv: <http://www.wikidata.org/prop/statement/value/>
            PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX wikibase: <http://wikiba.se/ontology#>
            PREFIX p: <http://www.wikidata.org/prop/>
            PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
            PREFIX ps: <http://www.wikidata.org/prop/statement/>
            PREFIX wd: <http://www.wikidata.org/entity/>
            PREFIX wdt: <http://www.wikidata.org/prop/direct/>

            SELECT ?wdt_required_property {{
              <{constraint_statement}> pq:P2306/wikibase:directClaim ?wdt_required_property.
              # exception and deprecation
              FILTER NOT EXISTS {{<{constraint_statement}> pq:P2241 []}}
              FILTER NOT EXISTS {{<{constraint_statement}> wikibase:rank wikibase:DeprecatedRank}}
            }}
            """
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        #print(response.text)

        # Parse the XML data
        namespace = {'sparql': 'http://www.w3.org/2005/sparql-results#'}
        root = ET.fromstring(response.text)

        # Find the uri inside the binding
        if root.find('.//sparql:binding[@name="wdt_required_property"]/sparql:uri', namespace) is not None:
            return root.find('.//sparql:binding[@name="wdt_required_property"]/sparql:uri', namespace).text
        return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
def hasRequiredPropertyChanged(row):
    constraint_id_2019 = getConstraintID(row)
    if constraint_id_2019 is None:
        return None
    req_prop_2023 = getRequiredProperty(constraint_id_2019)
    if req_prop_2023 is None:
        return False
    if req_prop_2023 != row['wdt_required_property']:
        return True
    return False
        
    
# Example usage 
#hasRequiredPropertyChanged(df_vrs_repairs_cleaned.iloc[61485])
hasRequiredPropertyChanged(df_vrs_repairs_cleaned.iloc[367])

In [ ]:
df_vrs_repairs_cleaned['CQ_replaced_property'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs_cleaned.iterrows(), total=len(df_vrs_repairs_cleaned)):
    
    # Check for unprocessed rows
    if pd.isna(row['CQ_replaced_property']):
        result = hasRequiredPropertyChanged(row)
        df_vrs_repairs_cleaned.at[index, 'CQ_replaced_property'] = result

    # Save a checkpoint every 10,000 rows
    if index % 500000 == 0:
        df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_vrs_repairs_cleaned['CQ_replaced_property'].value_counts()

In [ ]:
df_vrs_repairs_cleaned.iloc[367]

In [ ]:
df_vrs_repairs_cleaned[
    (df_vrs_repairs_cleaned['C_deleted'] == False)
    & (df_vrs_repairs_cleaned['C_deprecated'] == False)
    & (df_vrs_repairs_cleaned['CQ_added_exception'] == False)
    & (df_vrs_repairs_cleaned['S_deleted'] == False)
    & (df_vrs_repairs_cleaned['CQ_replaced_property'] == False)
    & ((df_vrs_repairs_cleaned['Sc_added'] == False) 
       | (pd.isna(df_vrs_repairs_cleaned['Sc_added'])))
]['2019_no_req_val'].value_counts()

In [ ]:
df_vrs_repairs_cleaned[
    (df_vrs_repairs_cleaned['C_deleted'] == False)
    & (df_vrs_repairs_cleaned['C_deprecated'] == False)
    & (df_vrs_repairs_cleaned['CQ_added_exception'] == False)
    & (df_vrs_repairs_cleaned['S_deleted'] == False)
    & (df_vrs_repairs_cleaned['CQ_replaced_property'] == False)
    & ((df_vrs_repairs_cleaned['Sc_added'] == False) 
       | (pd.isna(df_vrs_repairs_cleaned['Sc_added'])))
]

In [ ]:
getConstraintID(df_vrs_repairs_cleaned.iloc[473531])

- testing for conext statement addition (Sc_added):

In [ ]:
import requests
import xml.etree.ElementTree as ET

def reqValStmtAdded(row):
    
    if row['object'].startswith("_:"):
        return None
    
    if row['2019_no_req_val'] is True:
        return None
    
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX wikibase: <http://wikiba.se/ontology#>
        PREFIX p: <http://www.wikidata.org/prop/>
        PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
        PREFIX ps: <http://www.wikidata.org/prop/statement/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    
        ASK {{ 
          <{row['property']}> p:P2302 ?statement.
          ?statement ps:P2302 wd:Q21510864.
          ?statement pq:P2305 ?expected_val.
          <{row['object']}> <{row['wdt_required_property']}> ?expected_val.
        }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
print(reqValStmtAdded(df_vrs_repairs_cleaned.iloc[473531]))

In [ ]:
df_vrs_repairs_cleaned.to_csv("checkpoint_vrs_ok.csv", index=False)

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs_cleaned.iterrows(), total=len(df_vrs_repairs_cleaned)):
    
    # Check for unprocessed rows
    if bool(row['2019_no_req_val']) is False:
        result = reqValStmtAdded(row)
        df_vrs_repairs_cleaned.at[index, 'Sc_added'] = result

    # Save a checkpoint every 10,000 rows
    if index % 100000 == 0:
        df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_vrs_repairs_cleaned[(df_vrs_repairs_cleaned['Sc_added'] == True)] )

In [ ]:
df_vrs_repairs_cleaned[
    (df_vrs_repairs_cleaned['C_deleted'] == False)
    & (df_vrs_repairs_cleaned['C_deprecated'] == False)
    & (df_vrs_repairs_cleaned['CQ_added_exception'] == False)
    & (df_vrs_repairs_cleaned['S_deleted'] == False)
    & (df_vrs_repairs_cleaned['CQ_replaced_property'] == False)
    & ((df_vrs_repairs_cleaned['Sc_added'] == False) 
       | (pd.isna(df_vrs_repairs_cleaned['Sc_added'])))
]

- testing T-box required value changes (CQ_replaced_value):

In [ ]:
df_vrs_repairs_cleaned['CQ_replaced_value'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getCountRequiredValues(row, endpoint):

    if bool(row['2019_no_req_val']) and  endpoint == "ENTER_qEndpoint_WD_2019":
        return 0
    
    
    # SPARQL query
    query = f"""
            PREFIX psv: <http://www.wikidata.org/prop/statement/value/>
            PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX wikibase: <http://wikiba.se/ontology#>
            PREFIX p: <http://www.wikidata.org/prop/>
            PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
            PREFIX ps: <http://www.wikidata.org/prop/statement/>
            PREFIX wd: <http://www.wikidata.org/entity/>
            PREFIX wdt: <http://www.wikidata.org/prop/direct/>

            SELECT (COUNT(?expected_values) as ?total) {{
              ?statement ps:P2302 wd:Q21510864. ## value-requires-statement constraint
              <{row['property']}> p:P2302 ?statement.
              ?statement pq:P2306/wikibase:directClaim <{row['wdt_required_property']}>.
              # exception and deprecation
              FILTER NOT EXISTS {{?statement pq:P2241 []}}
              FILTER NOT EXISTS {{?statement wikibase:rank wikibase:DeprecatedRank}}
              ?statement pq:P2305 ?expected_values.
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:

        # Parse the XML data
        namespace = {'sparql': 'http://www.w3.org/2005/sparql-results#'}
        root = ET.fromstring(response.text)

        # Find the uri inside the binding
        if root.find('.//sparql:binding[@name="total"]/sparql:literal', namespace) is not None:
            return root.find('.//sparql:binding[@name="total"]/sparql:literal', namespace).text
        return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
def hasRequiredValuesNumberIncreased(row):
    n_2019 = getCountRequiredValues(row,"ENTER_qEndpoint_WD_2019")
    if n_2019 is None:
        return None
    n_2023 = getCountRequiredValues(row,"ENTER_qEndpoint_WD_2023")
    if n_2023 is None:
        return None
    if int(n_2023) > int(n_2019):
        return True
    return False
    
#print(getCountRequiredValues(df_vrs_repairs_cleaned.iloc[497101],"ENTER_qEndpoint_WD_2019"))
#print(getCountRequiredValues(df_vrs_repairs_cleaned.iloc[497101],"ENTER_qEndpoint_WD_2023"))
#hasRequiredValuesNumberIncreased(df_vrs_repairs_cleaned.iloc[497101])

In [ ]:
df_vrs_repairs_cleaned['CQ_replaced_value'] =  None

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs_cleaned.iterrows(), total=len(df_vrs_repairs_cleaned)):
    
    if pd.isna(row['CQ_replaced_value']) or row['CQ_replaced_value'] is None:
        if bool(row['C_deleted']) is True:
            df_vrs_repairs_cleaned.at[index, 'CQ_replaced_value'] = False
        else:
            result = hasRequiredValuesNumberIncreased(row)
            df_vrs_repairs_cleaned.at[index, 'CQ_replaced_value'] = result

    # Save a checkpoint every 10,000 rows
    if index % 100000 == 0 and index != 0:
        df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_vrs_repairs_cleaned[(df_vrs_repairs_cleaned['CQ_replaced_value'] == True)] )

In [ ]:
df_vrs_repairs_cleaned[
    (df_vrs_repairs_cleaned['C_deleted'] == False)
    & (df_vrs_repairs_cleaned['C_deprecated'] == False)
    & (df_vrs_repairs_cleaned['CQ_added_exception'] == False)
    & (df_vrs_repairs_cleaned['S_deleted'] == False)
    & (df_vrs_repairs_cleaned['CQ_replaced_value'] == False)
    & (df_vrs_repairs_cleaned['CQ_replaced_property'] == False)
    & ((df_vrs_repairs_cleaned['Sc_added'] == False) 
       | (pd.isna(df_vrs_repairs_cleaned['Sc_added'])))
]

In [ ]:
getConstraintID(df_vrs_repairs_cleaned.iloc[1296137])

In [ ]:
getConstraintID(df_vrs_repairs_cleaned.iloc[1296137], "ENTER_qEndpoint_WD_2023")

- revalidating deprecated ranks and deletions of constraints:

In [ ]:
import requests
import xml.etree.ElementTree as ET

def constraintDeprecated(constraint_id):
    
    
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{constraint_id}> ?p ?o 
            FILTER (?p = wikibase:rank)
            FILTER (?o = wikibase:DeprecatedRank)
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    #print(query)
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

def constraintDeleted(constraint_id):
    
    
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{constraint_id}> ?p ?o 
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    #print(query)
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
    
# Example usage
print(constraintDeprecated(getConstraintID(df_vrs_repairs_cleaned.iloc[1296137])))
print(constraintDeleted(getConstraintID(df_vrs_repairs_cleaned.iloc[1296137])))

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs_cleaned.iterrows(), total=len(df_vrs_repairs_cleaned)):
    
    constraint_id = getConstraintID(row)
    if bool(row['C_deprecated']) is False:
        result = constraintDeprecated(constraint_id)
        df_vrs_repairs_cleaned.at[index, 'C_deprecated'] = result
        
    if bool(row['C_deleted']) is False:
            result = constraintDeleted(constraint_id)
            df_vrs_repairs_cleaned.at[index, 'C_deleted'] = result

    # Save a checkpoint every 10,000 rows
    if index % 300000 == 0 and index != 0:
        df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs_cleaned.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['C_deprecated'] == True)] )


In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['C_deleted'] == True)] )

In [ ]:
df_vrs_repairs[
    (df_vrs_repairs['C_deleted'] == False)
    & (df_vrs_repairs['C_deprecated'] == False)
    & (df_vrs_repairs['CQ_added_exception'] == False)
    & (df_vrs_repairs['S_deleted'] == False)
    & (df_vrs_repairs['CQ_replaced_value'] == False)
    & (df_vrs_repairs['CQ_replaced_property'] == False)
    & ((df_vrs_repairs['Sc_added'] == False) 
       | (pd.isna(df_vrs_repairs['Sc_added'])))
]

In [ ]:
df_vrs_repairs.to_csv("vrs_repairs_all.csv", index=False)

In [ ]:
df_vrs_repairs.dtypes

In [ ]:
len(df_vrs_repairs[(df_vrs_repairs['Sc_added'] == True)] )

- testing for required value deletions (CQ_deleted_value):

In [ ]:
df_vrs_repairs['CQ_deleted_value'] = None

In [ ]:
def requiredValueRemoved(row):
    n_2019 = getCountRequiredValues(row,"ENTER_qEndpoint_WD_2019")
    if n_2019 is None:
        return None
    n_2023 = getCountRequiredValues(row,"ENTER_qEndpoint_WD_2023")
    if n_2023 is None:
        return None
    if int(n_2023) == 0 and int(n_2019) > 0:
        return True
    return False

In [ ]:
requiredValueRemoved(df_vrs_repairs.iloc[1345664])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_vrs_repairs.iterrows(), total=len(df_vrs_repairs)):
    
    if pd.isna(row['CQ_deleted_value']) or row['CQ_deleted_value'] is None:
        if bool(row['C_deleted']) is True:
            df_vrs_repairs.at[index, 'CQ_deleted_value'] = False
        else:
            result = requiredValueRemoved(row)
            df_vrs_repairs.at[index, 'CQ_deleted_value'] = result

    # Save a checkpoint every 10,000 rows
    if index % 100000 == 0 and index != 0:
        df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_vrs_repairs.to_csv("checkpoint_vrs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_vrs_repairs['CQ_deleted_value'].value_counts()

In [ ]:
df_vrs_repairs[
    (df_vrs_repairs['C_deleted'] == False)
    & (df_vrs_repairs['C_deprecated'] == False)
    & (df_vrs_repairs['CQ_added_exception'] == False)
    & (df_vrs_repairs['S_deleted'] == False)
    & (df_vrs_repairs['CQ_replaced_value'] == False)
    & (df_vrs_repairs['CQ_replaced_property'] == False)
    & ((df_vrs_repairs['Sc_added'] == False) 
       | (pd.isna(df_vrs_repairs['Sc_added'])))
]

In [ ]:
df_vrs_repairs.to_csv("vrs_all_repairs.csv", index=False)

- testing for base statement value replacement:

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getObjectsOfStatement(subject, property):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # Prepare the property
    pid = property.replace('http://www.wikidata.org/entity/', '')

    # SPARQL query
    query = f"""
    SELECT ?o {{
        <{subject}> wdt:{pid} ?o
    }}
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)

    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"

    # Send HTTP GET request
    headers = {"Accept": "application/sparql-results+xml"}

    response = requests.get(url, headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        root = ET.fromstring(response.text)
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Collect all uri and literal elements
        uris = [uri.text for uri in root.findall('.//ns:uri', namespace)]
        literals = [lit.text for lit in root.findall('.//ns:literal', namespace)]

        results = uris + literals  # Combine both types of results

        return results if results else None
    else:
        print("Error:", response.text)
        return None

In [ ]:
df_vrs_repairs['Sr_replacement'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET
from tqdm import tqdm

def getObjectsOfStatementsBatch(subjects, pid):
    endpoint = "ENTER_qEndpoint_WD_2023"
    
    # Build VALUES list
    values_clause = " ".join(f"<{s}>" for s in subjects)
    
    query = f"""
    SELECT ?subject ?o WHERE {{
      VALUES ?subject {{ {values_clause} }}
      ?subject wdt:{pid} ?o
    }}
    """
    
    encoded_query = requests.utils.quote(query)
    url = f"{endpoint}?query={encoded_query}"
    headers = {"Accept": "application/sparql-results+xml"}

    response = requests.get(url, headers=headers)

    if response.ok:
        root = ET.fromstring(response.text)
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}
        
        # Map: subject -> list of objects
        results = {}

        for result in root.findall('.//ns:result', namespace):
            subj = result.find('./ns:binding[@name="subject"]/ns:uri', namespace)
            obj = result.find('./ns:binding[@name="o"]/ns:uri', namespace) or \
                  result.find('./ns:binding[@name="o"]/ns:literal', namespace)
            if subj is not None and obj is not None:
                subj_uri = subj.text
                obj_value = obj.text
                results.setdefault(subj_uri, []).append(obj_value)

        return results
    else:
        print("Error:", response.text)
        return None

# --- Batch processing ---
batch_size = 50

# Group by property if you have multiple PIDs, otherwise just process
for pid_value in df_vrs_repairs['property'].unique():
    # Subset dataframe by property
    df_subset = df_vrs_repairs[df_vrs_repairs['property'] == pid_value]
    
    for i in tqdm(range(0, len(df_subset), batch_size)):
        batch = df_subset.iloc[i:i+batch_size]
        subjects = batch['subject'].tolist()

        results = getObjectsOfStatementsBatch(subjects, pid_value.replace('http://www.wikidata.org/entity/', ''))

        for index, row in batch.iterrows():
            subject_uri = row['subject']
            expected_object = row['object']
            object_list = results.get(subject_uri, None)

            if row['Sr_replacement'] is None:
                if row['A-box wdt statement Deleted'] is True:
                    df_vrs_repairs.at[index, 'Sr_replacement'] = False 
                else:
                    if object_list is None or expected_object in object_list:
                        df_vrs_repairs.at[index, 'Sr_replacement'] = False
                    else:
                        df_vrs_repairs.at[index, 'Sr_replacement'] = True

In [ ]:
df_vrs_repairs['Sr_replacement'].value_counts()

In [ ]:
df_vrs_repairs.to_csv("vrs_all_repairs.csv", index=False)